# Genotype–Phenotype Heterogeneity Exploration with `mlcroissant`
This notebook demonstrates how to load, examine, and process the FAIRˆ dataset for Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency using the `mlcroissant` library.

### Dataset Source
The dataset is structured according to a Croissant schema and accessible from:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIRˆ dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\n\nDescription: {metadata.description}\n\nID: {metadata.id}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

This section shows all record sets defined in the dataset (by their `@id`), and lists out the available fields (columns) in each.


In [ ]:
# List all record sets by their @id (using Croissant metadata)
record_sets = dataset.metadata.record_sets
print("Record sets found:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}, name: {getattr(rs, 'name', 'Unnamed')}")

# List all fields/columns for each record set
record_set_fields = {}
for rs in record_sets:
    rs_fields = [f.id for f in rs.fields]
    record_set_fields[rs.id] = rs_fields
    print(f"\nFields in RecordSet {rs.id}:")
    for f in rs.fields:
        print(f"  - Field @id: {f.id}, name: {getattr(f, 'name', 'Unnamed')}, dataType: {getattr(f, 'data_type', '')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

We use the record set and field `@id`s from the overview to structure our exploration.

In [ ]:
# Extract data from each record set
dataframes = {}
for rs in record_sets:
    print(f"\nLoading records from RecordSet: {rs.id}")
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"Columns (@id): {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"(No records found for {rs.id})")

## 4. Exploratory Data Analysis (EDA)
Process, filter, and analyze data for a selected record set.

- Removes outliers
- Normalizes numeric values
- Groups and summarizes by categorical fields

*Note*: Entities referenced by their `@id`, as in Croissant metadata, see above for their list.

In [ ]:
# Choose one record set for demonstration
chosen_rs_id = None
for rs_id, df in dataframes.items():
    # Pick the largest non-empty table
    if chosen_rs_id is None or len(df) > len(dataframes[chosen_rs_id]):
        chosen_rs_id = rs_id

df = dataframes[chosen_rs_id]
print(f"Using RecordSet @id: {chosen_rs_id} for EDA")

# Find a numeric field (@id) for analysis
numeric_field_id = None
group_field_id = None
for f in dataset.metadata.record_sets:
    if f.id == chosen_rs_id:
        for field in f.fields:
            dtype = getattr(field, 'data_type', None)
            if dtype in ['Float', 'Integer', 'Number'] and numeric_field_id is None:
                numeric_field_id = field.id
            if dtype in ['Text', 'Boolean'] and group_field_id is None:
                group_field_id = field.id
        break

if numeric_field_id is None:
    print("No numeric field found in this record set.")
else:
    print(f"Numeric field for EDA: {numeric_field_id}")

    # Remove outliers: filter records above threshold
    threshold = df[numeric_field_id].mean() + 2 * df[numeric_field_id].std()
    filtered_df = df[df[numeric_field_id] < threshold]
    print(f"Filtered records where {numeric_field_id} < {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized values for {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by a categorical field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean of {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
# Visualize numeric field distribution
if numeric_field_id is not None:
    plt.figure(figsize=(6,4))
    filtered_df[numeric_field_id].hist(bins=10, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field exists, show mean bar chart
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(6,4))
        plt.bar(grouped_df[group_field_id].astype(str), grouped_df[numeric_field_id], color='orange')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook guided you through the process of loading structured clinical genotype–phenotype data with `mlcroissant`, exploring the available fields and records by their `@id`, extracting specific record sets, and applying simple processing and visualization. You can further extend this analysis to model relationships, compare across patients, or validate hypotheses for rare disease research.